# Lecture 18: Regularization And Pipelines

This notebook tunes regularized linear models inside a preprocessing pipeline.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
housing = pd.read_csv(DATA / "housing_sales.csv")
features = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit", "district"]
target = "price_k_eur"
X = housing[features]
y = housing[target]


In [ ]:
numeric = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit"]
categorical = ["district"]
preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ]
)

ridge_pipe = Pipeline(
    [
        ("preprocess", preprocess),
        ("model", Ridge()),
    ]
)


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_grid = GridSearchCV(
    ridge_pipe,
    param_grid={"model__alpha": [0.1, 1.0, 10.0, 100.0, 500.0]},
    cv=cv,
    scoring="neg_root_mean_squared_error",
)
ridge_grid.fit(X, y)
print(ridge_grid.best_params_)
print(f"Best CV RMSE: {-ridge_grid.best_score_:.2f}")


In [ ]:
lasso_pipe = Pipeline(
    [
        ("preprocess", preprocess),
        ("model", Lasso(max_iter=20000)),
    ]
)
lasso_grid = GridSearchCV(
    lasso_pipe,
    param_grid={"model__alpha": [0.01, 0.1, 1.0, 5.0, 10.0]},
    cv=cv,
    scoring="neg_root_mean_squared_error",
)
lasso_grid.fit(X, y)
print(lasso_grid.best_params_)
print(f"Best CV RMSE: {-lasso_grid.best_score_:.2f}")


In [ ]:
best_lasso = lasso_grid.best_estimator_
feature_names = best_lasso.named_steps["preprocess"].get_feature_names_out()
coefficients = pd.Series(best_lasso.named_steps["model"].coef_, index=feature_names)
coefficients.sort_values(key=abs, ascending=False)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
final_model = ridge_grid.best_estimator_
final_model.fit(X_train, y_train)
test_pred = final_model.predict(X_test)
print(f"Final test RMSE: {mean_squared_error(y_test, test_pred) ** 0.5:.2f}")


## LLM Check

Ask an LLM where scaling and one-hot encoding belong in a regularized workflow. Keep only the answer that places preprocessing inside cross-validation.
